# Anvikshaa— Face Recognition Training
**LBPH Face Recognizer — 4-step pipeline**

| Step | What it does |
|---|---|
| 1. Capture | Take face photos from webcam |
| 2. Augment | Create 5× variants per photo (flip, bright, dark, blur, noise) |
| 3. Train | Train LBPH model → saves `face_model.xml` |
| 4. Evaluate | Live webcam confidence test |

> Run cells **top to bottom** in order. Skip Cell 1 if you already have photos.

## Cell 1 — Setup & Dataset Status

In [1]:
import cv2, numpy as np, os, sys, pickle

# Paths — works from both root and notebooks/ folder
BASE_DIR   = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
FACES_DIR  = os.path.join(BASE_DIR, "authorized_faces")
MODEL_PATH = os.path.join(BASE_DIR, "face_model.xml")
NAMES_PATH = os.path.join(BASE_DIR, "name_dict.pkl")
IMG_SIZE   = (100, 100)
MIN_IMAGES = 30
CONFIDENCE_THRESHOLD = 80

# Load Haar Cascade
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

print(f"Working directory : {BASE_DIR}")
print(f"Faces folder      : {FACES_DIR}")
print(f"Model path        : {MODEL_PATH}")
print()

# Dataset status
print("── Dataset Status ──────────────────────────────────")
if os.path.exists(FACES_DIR):
    total = 0
    for person in sorted(os.listdir(FACES_DIR)):
        person_dir = os.path.join(FACES_DIR, person)
        if not os.path.isdir(person_dir):
            continue
        imgs      = [f for f in os.listdir(person_dir) if f.lower().endswith(('.jpg','.png','.jpeg'))]
        originals = [f for f in imgs if '_aug' not in f]
        augmented = [f for f in imgs if '_aug' in f]
        status    = '✅' if len(imgs) >= MIN_IMAGES else '⚠ needs more'
        total    += len(imgs)
        print(f"  {person.upper():<15} {len(originals):>3} originals  {len(augmented):>3} augmented  = {len(imgs):>3} total  {status}")
    model_ok = '✅ exists' if os.path.exists(MODEL_PATH) else '❌ not trained yet'
    print(f"\n  Total images    : {total}")
    print(f"  Model status    : {model_ok}")
else:
    print(f"  ❌ {FACES_DIR} not found — create it and add person subfolders")

Working directory : c:\Users\Harsh Rathod\OneDrive\Desktop\mini
Faces folder      : c:\Users\Harsh Rathod\OneDrive\Desktop\mini\authorized_faces
Model path        : c:\Users\Harsh Rathod\OneDrive\Desktop\mini\face_model.xml

── Dataset Status ──────────────────────────────────
  CHAITANYA        54 originals  270 augmented  = 324 total  ✅
  HARSH            60 originals  300 augmented  = 360 total  ✅
  RASHID            3 originals   15 augmented  =  18 total  ⚠ needs more
  SAISH             3 originals   15 augmented  =  18 total  ⚠ needs more

  Total images    : 720
  Model status    : ✅ exists


## Cell 2 — Step 1: Capture Face Photos (webcam)
> Skip this cell if you already have photos in `authorized_faces/<name>/`

In [7]:
# ── CONFIGURE ─────────────────────────────────────────────────
CAPTURE_NAME = "HARSH"   # ← change to the person's name
CAPTURE_N    = 50        # number of photos to take
# ──────────────────────────────────────────────────────────────

save_dir = os.path.join(FACES_DIR, CAPTURE_NAME.lower())
os.makedirs(save_dir, exist_ok=True)

existing  = len([f for f in os.listdir(save_dir) if f.lower().endswith(('.jpg','.png','.jpeg'))])
start_idx = existing

print(f"Capturing for    : {CAPTURE_NAME}")
print(f"Saving to        : {save_dir}")
print(f"Existing photos  : {existing}")
print(f"Will capture     : {CAPTURE_N} more")
print()
print("Controls:  SPACE = capture frame   |   Q = quit")
print("Tips: Look straight → tilt left → right → up → down\n")

cam   = cv2.VideoCapture(0)
count = 0

while count < CAPTURE_N:
    ret, frame = cam.read()
    if not ret:
        break

    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

    cv2.putText(frame, f"Person : {CAPTURE_NAME}",    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    cv2.putText(frame, f"Saved  : {count}/{CAPTURE_N}",(10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    face_txt = f"Face detected ({len(faces)})" if len(faces) > 0 else "NO FACE — move closer"
    cv2.putText(frame, face_txt, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (0,255,0) if len(faces) > 0 else (0,0,255), 2)
    cv2.putText(frame, "SPACE=capture  Q=quit", (10, frame.shape[0]-15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)

    cv2.imshow(f"Capture — {CAPTURE_NAME}", frame)
    key = cv2.waitKey(1)

    if key == ord(' '):
        if len(faces) == 0:
            print("  No face detected — skipping")
            continue
        x, y, w, h = faces[0]
        face_crop  = cv2.resize(gray[y:y+h, x:x+w], IMG_SIZE)
        filename   = os.path.join(save_dir, f"{CAPTURE_NAME.lower()}_{start_idx+count:03d}.jpg")
        cv2.imwrite(filename, face_crop)
        count += 1
        print(f"  [{count:02d}/{CAPTURE_N}] Saved → {os.path.basename(filename)}")
    elif key == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()
print(f"\n✅ Captured {count} photos for {CAPTURE_NAME}")
if existing + count < MIN_IMAGES:
    print(f"⚠ Recommended: at least {MIN_IMAGES} photos. Run Step 2 (augment) to expand dataset.")

Capturing for    : HARSH
Saving to        : c:\Users\Harsh Rathod\OneDrive\Desktop\mini\authorized_faces\harsh
Existing photos  : 360
Will capture     : 50 more

Controls:  SPACE = capture frame   |   Q = quit
Tips: Look straight → tilt left → right → up → down

  [01/50] Saved → harsh_360.jpg
  [02/50] Saved → harsh_361.jpg
  [03/50] Saved → harsh_362.jpg
  [04/50] Saved → harsh_363.jpg
  [05/50] Saved → harsh_364.jpg
  [06/50] Saved → harsh_365.jpg
  [07/50] Saved → harsh_366.jpg
  No face detected — skipping
  No face detected — skipping
  No face detected — skipping
  No face detected — skipping
  No face detected — skipping
  No face detected — skipping
  [08/50] Saved → harsh_367.jpg
  [09/50] Saved → harsh_368.jpg
  [10/50] Saved → harsh_369.jpg
  [11/50] Saved → harsh_370.jpg
  [12/50] Saved → harsh_371.jpg
  [13/50] Saved → harsh_372.jpg
  [14/50] Saved → harsh_373.jpg
  [15/50] Saved → harsh_374.jpg
  [16/50] Saved → harsh_375.jpg
  [17/50] Saved → harsh_376.jpg
  [18/50] Sav

## Cell 3 — Step 2: Augment Dataset (5× expansion)
Creates 5 variants per original: flip, bright, dark, blur, noise

In [8]:
def _add_noise(img):
    noise = np.random.normal(0, 15, img.shape).astype(np.int16)
    return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

total_new = 0
print("── Augmenting Dataset ──────────────────────────────")

for person in sorted(os.listdir(FACES_DIR)):
    person_dir = os.path.join(FACES_DIR, person)
    if not os.path.isdir(person_dir):
        continue

    originals = [
        f for f in os.listdir(person_dir)
        if f.lower().endswith(('.jpg','.png','.jpeg')) and '_aug' not in f
    ]

    print(f"  [{person.upper()}]  {len(originals)} originals", end="")
    new_count = 0

    for fname in originals:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img  = cv2.resize(img, IMG_SIZE)
        stem = os.path.splitext(fname)[0]

        augmentations = {
            "aug_flip":   cv2.flip(img, 1),
            "aug_bright": np.clip(img.astype(np.int16) + 40, 0, 255).astype(np.uint8),
            "aug_dark":   np.clip(img.astype(np.int16) - 40, 0, 255).astype(np.uint8),
            "aug_blur":   cv2.GaussianBlur(img, (5, 5), 0),
            "aug_noise":  _add_noise(img),
        }
        for aug_name, aug_img in augmentations.items():
            out_path = os.path.join(person_dir, f"{stem}_{aug_name}.jpg")
            if not os.path.exists(out_path):
                cv2.imwrite(out_path, aug_img)
                new_count += 1

    total_after = len([f for f in os.listdir(person_dir) if f.lower().endswith(('.jpg','.png','.jpeg'))])
    total_new  += new_count
    print(f"  >>  +{new_count} augmented  =  {total_after} total")

print(f"\n✅ Augmentation complete. {total_new} new images created.")

── Augmenting Dataset ──────────────────────────────
  [CHAITANYA]  54 originals  >>  +0 augmented  =  324 total
  [HARSH]  110 originals  >>  +250 augmented  =  660 total
  [RASHID]  3 originals  >>  +0 augmented  =  18 total
  [SAISH]  3 originals  >>  +0 augmented  =  18 total

✅ Augmentation complete. 250 new images created.


## Cell 4 — Step 3: Train LBPH Model

In [9]:
faces, labels, label_map = [], [], {}
idx = 0

print("── Loading Training Images ─────────────────────────")
for person in sorted(os.listdir(FACES_DIR)):
    person_dir = os.path.join(FACES_DIR, person)
    if not os.path.isdir(person_dir):
        continue

    imgs_loaded = 0
    for fname in os.listdir(person_dir):
        if not fname.lower().endswith(('.jpg','.png','.jpeg')):
            continue
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        faces.append(cv2.resize(img, IMG_SIZE))
        labels.append(idx)
        imgs_loaded += 1

    if imgs_loaded > 0:
        label_map[idx] = person.upper()
        print(f"    [{idx}] {person.upper():<15} — {imgs_loaded} images")
        idx += 1

print(f"\n  Total images : {len(faces)}")
print(f"  Persons      : {len(label_map)}")

# Train
print(f"  Training LBPH model...", end=" ", flush=True)
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.train(faces, np.array(labels))
recognizer.save(MODEL_PATH)

with open(NAMES_PATH, 'wb') as f:
    pickle.dump(label_map, f)

print("Done!")
print(f"  Model saved  : {MODEL_PATH}")
print(f"  Names saved  : {NAMES_PATH}")
print(f"  Authorized   : {list(label_map.values())}")
print(f"\n✅ Training complete! Run main.py to start the system.")

── Loading Training Images ─────────────────────────
    [0] CHAITANYA       — 324 images
    [1] HARSH           — 660 images
    [2] RASHID          — 18 images
    [3] SAISH           — 18 images

  Total images : 1020
  Persons      : 4
  Training LBPH model... Done!
  Model saved  : c:\Users\Harsh Rathod\OneDrive\Desktop\mini\face_model.xml
  Names saved  : c:\Users\Harsh Rathod\OneDrive\Desktop\mini\name_dict.pkl
  Authorized   : ['CHAITANYA', 'HARSH', 'RASHID', 'SAISH']

✅ Training complete! Run main.py to start the system.


## Cell 5 — Step 4: Evaluate — Live Webcam Test
> Shows recognition name + confidence score in real time.  
> **Confidence < 80 = recognized ✅ | > 80 = unknown ❌**  
> Press **Q** to stop.

In [10]:
# Load the trained model
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read(MODEL_PATH)

with open(NAMES_PATH, 'rb') as f:
    label_map = pickle.load(f)

print(f"Model loaded   : {MODEL_PATH}")
print(f"Authorized     : {list(label_map.values())}")
print(f"Threshold      : < {CONFIDENCE_THRESHOLD} = recognized")
print("Press Q to quit\n")

cam = cv2.VideoCapture(0)
recognized_count = unknown_count = total_frames = 0

while True:
    ret, frame = cam.read()
    if not ret:
        break

    total_frames += 1
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))

    for (x, y, w, h) in faces:
        face_roi = cv2.resize(gray[y:y+h, x:x+w], IMG_SIZE)
        try:
            label, confidence = recognizer.predict(face_roi)
            if confidence < CONFIDENCE_THRESHOLD:
                name  = label_map.get(label, "UNKNOWN")
                color = (0, 255, 0)  # Green
                tag   = f"{name}  conf:{confidence:.1f}"
                recognized_count += 1
            else:
                name  = "UNKNOWN"
                color = (0, 0, 255)  # Red
                tag   = f"UNKNOWN  conf:{confidence:.1f}"
                unknown_count += 1
        except Exception as e:
            tag, color = "ERROR", (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.rectangle(frame, (x, y-32), (x+w, y), color, cv2.FILLED)
        cv2.putText(frame, tag, (x+4, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)

    acc = recognized_count / max(recognized_count + unknown_count, 1) * 100
    cv2.putText(frame, f"Frames:{total_frames}  Recognized:{recognized_count}  Unknown:{unknown_count}  Acc:{acc:.0f}%",
                (10, frame.shape[0]-15), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (200,200,200), 1)

    cv2.imshow("Anviksha — Face Recognition Evaluation", frame)
    if cv2.waitKey(1) == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

print(f"\n── Evaluation Results ──────────────")
print(f"  Frames tested   : {total_frames}")
print(f"  Recognized      : {recognized_count}")
print(f"  Unknown         : {unknown_count}")
print(f"  Recognition rate: {acc:.1f}%")

Model loaded   : c:\Users\Harsh Rathod\OneDrive\Desktop\mini\face_model.xml
Authorized     : ['CHAITANYA', 'HARSH', 'RASHID', 'SAISH']
Threshold      : < 80 = recognized
Press Q to quit


── Evaluation Results ──────────────
  Frames tested   : 20174
  Recognized      : 17623
  Unknown         : 3144
  Recognition rate: 84.9%
